# 指标库整理

本Notebook用于整理和计算各类金融指标，包括融资成本测算等。

## 1. 环境配置与依赖导入

In [ ]:
# 导入标准库
import os
import sys
import datetime

# 导入数据处理库
import pandas as pd
import numpy as np

# 导入机器学习库
from sklearn.linear_model import LinearRegression

# 导入可视化库
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 导入数据库库
import sqlalchemy
from sqlalchemy import create_engine
import psycopg2

# 导入配置
from config import DATABASE_URL, POSTGRESQL_URL, FINANCING_COST_MIN, FINANCING_COST_MAX, PROJECT_NAME

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print(f"项目: {PROJECT_NAME}")
print(f"当前时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 数据库连接

In [ ]:
# MySQL连接
def get_mysql_connection():
    """获取MySQL数据库连接"""
    try:
        engine = create_engine(DATABASE_URL, poolclass=sqlalchemy.pool.NullPool)
        conn = engine.connect()
        print("MySQL数据库连接成功")
        return engine, conn
    except Exception as e:
        print(f"MySQL数据库连接失败: {e}")
        return None, None

# PostgreSQL连接
def get_postgres_connection():
    """获取PostgreSQL数据库连接"""
    try:
        engine = create_engine(POSTGRESQL_URL)
        conn = engine.connect()
        print("PostgreSQL数据库连接成功")
        return engine, conn
    except Exception as e:
        print(f"PostgreSQL数据库连接失败: {e}")
        return None, None

mysql_engine, mysql_conn = get_mysql_connection()
pg_engine, pg_conn = get_postgres_connection()

## 3. 有息债务规模计算

In [ ]:
def calculate_interest_bearing_debt(df_indicator):
    """
    计算有息债务规模
    
    参数:
        df_indicator: 指标数据DataFrame，包含dt, trade_code, 指标, value列
    
    返回:
        计算后的DataFrame
    """
    # 短期债务相关指标
    st_indicators = [
        'ths_st_borrow_bond', 'ths_borrowing_funds_bond', 
        'ths_tradable_fnncl_liab_bond', 'ths_derivative_fnncl_liab_bond',
        'ths_bill_payable_bond', 'ths_fnncl_assets_sold_for_repur_bond',
        'ths_noncurrent_liab_due_in1y_bond', 'ths_st_bond_payable_new_bond'
    ]
    
    # 长期债务相关指标
    lt_indicators = [
        'ths_lt_loan_bond', 'ths_bond_payable_bond', 
        'ths_perpetual_capital_sec_bond', 'ths_preferred_bond', 
        'ths_lease_libilities_bond'
    ]
    
    # 按主体聚合计算
    aggregated = df_indicator.groupby(['dt', 'trade_code']).agg({
        'value': lambda x: x.sum() if x.name in st_indicators + lt_indicators else x.max()
    }).reset_index()
    
    # 计算有息债务
    # 这里简化处理，实际应根据业务逻辑详细计算
    result = aggregated.copy()
    result['指标'] = '有息债务'
    
    return result

print("有息债务计算函数已定义")

## 4. 融资成本计算

In [ ]:
def linear_extrapolation(group, cost_min=2.0, cost_max=8.0):
    """
    使用线性回归进行融资成本插值
    
    参数:
        group: 分组数据
        cost_min: 融资成本最小值(%)
        cost_max: 融资成本最大值(%)
    
    返回:
        插值后的数据
    """
    group = group.sort_values(by='债券融资成本')
    mask = np.isnan(group['融资成本'])
    
    if not mask.any():
        return group
    
    try:
        x = group.loc[~mask, ['债券融资成本']]
        y = group.loc[~mask, '融资成本']
        
        if len(x) > 1:
            model = LinearRegression().fit(x, y)
            if 0 < model.coef_[0] < 1:
                group.loc[mask, '融资成本'] = model.predict(group.loc[mask, ['债券融资成本']])
            else:
                # 使用简单线性插值
                c = y.iloc[0]
                bond_cost = x.iloc[0, 0]
                group.loc[mask, '融资成本'] = (group.loc[mask, '债券融资成本'] - bond_cost) / 3 + c
        elif len(x) == 1:
            c = y.iloc[0]
            bond_cost = x.iloc[0, 0]
            group.loc[mask, '融资成本'] = (group.loc[mask, '债券融资成本'] - bond_cost) / 3 + c
        else:
            group.loc[mask, '融资成本'] = group.loc[mask, '债券融资成本']
        
        # 限制融资成本范围
        group.loc[group['融资成本'] < cost_min, '融资成本'] = np.nan
        group.loc[group['融资成本'] > cost_max, '融资成本'] = np.nan
        
    except Exception as e:
        print(f"插值计算错误: {e}")
    
    return group

print("融资成本插值函数已定义")

In [ ]:
# 示例：计算融资成本
if pg_conn is not None:
    # 从数据库读取数据
    query = """
    SELECT * FROM 融资成本1 
    WHERE 融资成本 > 0 OR 债券融资成本 > 0
    """
    
    try:
        df = pd.read_sql(query, pg_conn)
        print(f"读取到 {len(df)} 条记录")
        
        if not df.empty:
            # 数据预处理
            df = df[df['债券融资成本'].notna()]
            df = df[(df['融资成本'].isna()) | ((df['融资成本'] <= 0.2) & (df['融资成本'] >= 0.03))]
            df = df[df['债券融资成本'] <= 20]
            df['融资成本'] = df['融资成本'] * 100
            
            # 按trade_code分组应用插值
            df = df[df.groupby('trade_code')['融资成本'].transform('any')]
            df_result = df.groupby('trade_code').apply(
                lambda g: linear_extrapolation(g, FINANCING_COST_MIN, FINANCING_COST_MAX)
            )
            
            print(f"\n插值计算完成，结果数据量: {len(df_result)}")
            display(df_result.head())
    except Exception as e:
        print(f"读取数据失败: {e}")
        df_result = None
else:
    print("PostgreSQL未连接，跳过融资成本计算")
    df_result = None

## 5. 融资成本可视化分析

In [ ]:
# 融资成本与债券融资成本关系图
if df_result is not None and not df_result.empty:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # 过滤有效数据
    plot_data = df_result[(df_result['融资成本'].notna()) & (df_result['债券融资成本'].notna())]
    plot_data = plot_data[(plot_data['融资成本'] <= 8) & (plot_data['债券融资成本'] <= 10)]
    
    ax.scatter(plot_data['债券融资成本'], plot_data['融资成本'], alpha=0.5, s=10)
    
    # 添加对角线
    ax.plot([0, 10], [0, 10], 'r--', label='y=x')
    
    ax.set_xlabel('债券融资成本 (%)')
    ax.set_ylabel('融资成本 (%)')
    ax.set_title('融资成本 vs 债券融资成本')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    os.makedirs('output', exist_ok=True)
    plt.savefig('output/financing_cost_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 融资成本分布直方图
if df_result is not None and not df_result.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 融资成本分布
    valid_fc = df_result['融资成本'].dropna()
    axes[0].hist(valid_fc, bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('融资成本 (%)')
    axes[0].set_ylabel('频数')
    axes[0].set_title('融资成本分布')
    axes[0].axvline(valid_fc.mean(), color='red', linestyle='--', label=f'均值: {valid_fc.mean():.2f}%')
    axes[0].legend()
    
    # 债券融资成本分布
    valid_bfc = df_result['债券融资成本'].dropna()
    axes[1].hist(valid_bfc, bins=50, edgecolor='black', alpha=0.7, color='orange')
    axes[1].set_xlabel('债券融资成本 (%)')
    axes[1].set_ylabel('频数')
    axes[1].set_title('债券融资成本分布')
    axes[1].axvline(valid_bfc.mean(), color='red', linestyle='--', label=f'均值: {valid_bfc.mean():.2f}%')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('output/financing_cost_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. 数据库存储

In [ ]:
# 将计算结果存回数据库
if df_result is not None and not df_result.empty and pg_conn is not None:
    try:
        df_result.to_sql('融资成本2', con=pg_conn, if_exists='replace', index=False)
        print(f"融资成本数据已存入数据库，共 {len(df_result)} 条记录")
    except Exception as e:
        print(f"存储融资成本数据失败: {e}")

## 7. 结果输出与总结

In [ ]:
# 生成输出报告
print("\n" + "="*60)
print(f"项目: {PROJECT_NAME} - 分析报告")
print("="*60)

if df_result is not None and not df_result.empty:
    print(f"\n数据处理完成:")
    print(f"  - 处理记录数: {len(df_result)}")
    print(f"  - 主体数量: {df_result['trade_code'].nunique()}")
    
    # 统计信息
    valid_fc = df_result['融资成本'].dropna()
    print(f"\n融资成本统计:")
    print(f"  - 有效记录数: {len(valid_fc)}")
    print(f"  - 平均值: {valid_fc.mean():.2f}%")
    print(f"  - 中位数: {valid_fc.median():.2f}%")
    print(f"  - 标准差: {valid_fc.std():.2f}%")
    
    # 保存结果
    output_file = 'output/financing_cost_result.xlsx'
    os.makedirs('output', exist_ok=True)
    df_result.to_excel(output_file, index=False)
    print(f"\n结果已保存至: {output_file}")
else:
    print("\n无数据可处理")

print("\n" + "="*60)
print("分析完成！")
print("="*60)

In [ ]:
# 关闭数据库连接
if mysql_conn is not None:
    mysql_conn.close()
    print("MySQL连接已关闭")

if pg_conn is not None:
    pg_conn.close()
    print("PostgreSQL连接已关闭")